### Fine tune LLM embeedings model


In [2]:
#!pip install openai==0.28.1
#!pip install openai --upgrade
#!pip install ragas
#!pip install unstructured
#!pip install langchain[all]
#!pip install --upgrade langchain

#!pip install playwright
#!pip install -U selenium unstructured
#!pip install --upgrade langchain langchain-community langchainhub langchain-openai langchain-chroma bs4

In [1]:
#!pip install pydantic==2.5
#!pip install --upgrade --quiet  langchain_milvus
#!pip install --upgrade scipy

In [1]:
#!pip install -U langchain-openai

In [1]:
#!pip install llama-index-embeddings-langchain

In [1]:
import os, json
import pandas as pd
#import openai
#from langchain.chat_models import ChatOpenAI, ChatGooglePalm
#from utils import OPENAI_API_KEY

#from Contextual_RAG.naive_rag import NaiveRAG 
from Contextual_RAG.contextual_rag import ContextualRAG


#os.environ['OPENAI_API_KEY'] =  OPENAI_API_KEY
#os.environ["LANGCHAIN_TRACING_V2"] = "true"
#os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

#openai.api_key = os.environ['OPENAI_API_KEY']

In [2]:
## Initialize and Train Both RAG Systems

# Initialize both RAG systems with the same embedding model for fair comparison
embedding_model = "BAAI/bge-base-en-v1.5"
chunk_size = 2000
chunk_overlap = 200


contextual_rag = ContextualRAG(
    embedding_model_name=embedding_model,
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    context_window= 5
)

Using device: cuda
Loading embedding model: BAAI/bge-base-en-v1.5


In [55]:
#Physics Context
physics_srt="""
Our laboratory uses advanced magnetic resonance techniques alongside precise radiation dose measurements to better understand material properties.
Researchers combine nuclear imaging with computed tomography to map the distribution of radioactive isotopes and quantify the radiation dose absorbed by samples.
In high-energy experiments, particle therapy methods are applied while nuclear imaging and computed tomography are used to track transient radiation dose effects.
The synergy between magnetic resonance and computed tomography enhances our ability to characterize materials and accurately measure radiation dose in complex experiments.
By integrating nuclear imaging with magnetic resonance methods, we can simultaneously monitor particle therapy beams and record the associated radiation dose.
Our experimental setup employs computed tomography and nuclear imaging to provide detailed spatial resolution while we control the radiation dose during particle therapy experiments.
Advances in magnetic resonance spectroscopy and nuclear imaging enable us to precisely calibrate the radiation dose delivered during high-energy physics studies.
Particle therapy research benefits from combining computed tomography and magnetic resonance techniques to visualize beam interactions and assess the localized radiation dose.
In our study of plasma dynamics, computed tomography and nuclear imaging are used together to capture rapid changes in radiation dose, with magnetic resonance providing complementary data.
The integration of particle therapy concepts with magnetic resonance and nuclear imaging techniques allows us to explore innovative ways of measuring radiation dose in harsh experimental conditions.
"""

#Medicine Context: 
medicine_str="""
In modern oncology, accurate radiation dose delivery during particle therapy is often guided by magnetic resonance imaging to ensure precise tumor targeting.
Clinicians routinely combine computed tomography with nuclear imaging to evaluate both anatomical details and functional aspects when planning a treatment strategy that minimizes radiation dose.
Magnetic resonance imaging is used alongside computed tomography to refine treatment planning, ensuring that the prescribed radiation dose for radiotherapy is both accurate and safe.
The integration of nuclear imaging and magnetic resonance techniques helps physicians monitor radiation dose in real time during advanced particle therapy sessions.
In pediatric radiology, computed tomography protocols are optimized by combining nuclear imaging feedback with meticulous radiation dose control to reduce overall exposure.
Physicians employ computed tomography and nuclear imaging together to assess disease progression while carefully adjusting the radiation dose during particle therapy.
An enhanced magnetic resonance imaging system is frequently paired with particle therapy to better control and measure the radiation dose delivered to cancerous tissues.
Hybrid imaging approaches that merge computed tomography with nuclear imaging allow for improved anatomical and functional assessment, which is critical in managing radiation dose during treatment.
Adaptive radiotherapy often uses magnetic resonance imaging together with computed tomography, enabling doctors to personalize the radiation dose based on real-time tumor responses.
Advanced treatment planning now integrates nuclear imaging, magnetic resonance, and computed tomography in order to optimize particle therapy protocols and maintain an ideal radiation dose for each patient.
"""

In [47]:
#Physics Context
physics_srt="""
Our laboratory uses advanced magnetic resonance techniques alongside precise radiation dose measurements to better understand material properties.
Researchers combine nuclear imaging with computed tomography to map the distribution of radioactive isotopes and quantify the radiation dose absorbed by samples.
"""

#Medicine Context: 
medicine_str="""
In modern oncology, accurate radiation dose delivery during particle therapy is often guided by magnetic resonance imaging to ensure precise tumor targeting.
Clinicians routinely combine computed tomography with nuclear imaging to evaluate both anatomical details and functional aspects when planning a treatment strategy that minimizes radiation dose.
"""

In [56]:
demo_dict = {'physics': {'input':physics_srt},
             'medicine': {'input': medicine_str}
            }

out = contextual_rag.demo_contextual_rag([demo_dict['physics']['input']])

demo_dict['physics']['output'] = out

out = contextual_rag.demo_contextual_rag([demo_dict['medicine']['input']])
demo_dict['medicine']['output'] = out

Extracting keywords: 100%|████████████████████████| 1/1 [00:00<00:00, 55.77it/s]


Extracted 15 unique keywords across 1 chunks


Generating chunk embeddings: 100%|████████████████| 1/1 [00:00<00:00, 12.33it/s]


Generated token embeddings for 1 chunks
Generated token IDs for 1 chunks
Token embedding shape: torch.Size([259, 768]) and Type: <class 'torch.Tensor'>
Generated 1 chunk embeddings using mean pooling


Generating keyword embeddings: 100%|██████████████| 1/1 [00:00<00:00, 80.36it/s]


...

Generated token embeddings for 1 chunks
Generated token IDs for 1 chunks
Token embedding shape: torch.Size([282, 768]) and Type: <class 'torch.Tensor'>
Generated 1 chunk embeddings using mean pooling


Generating keyword embeddings: 100%|█████████████| 1/1 [00:00<00:00, 116.13it/s]


Generated embeddings for 86 keyword-context-document pairs
Average 5.73 contexts per keyword


Generating chunk embeddings: 100%|████████████████| 1/1 [00:00<00:00, 79.54it/s]

Generated embeddings for 1 chunks
Found keywords in 1 chunks
Average 86.00 keywords per chunk with keywords


In [57]:
k_names = "keywords", "keywords_embeddings", "chunk_embeddings",  "averaged_keywords_embeddings"
for k in demo_dict:
    out = demo_dict[k]['output']
    demo_dict[k]['output'] = {}
    for i,name in enumerate(k_names):
        demo_dict[k]['output'][name] = out[i]

In [58]:
print(demo_dict['physics']['input'])


Our laboratory uses advanced magnetic resonance techniques alongside precise radiation dose measurements to better understand material properties.
Researchers combine nuclear imaging with computed tomography to map the distribution of radioactive isotopes and quantify the radiation dose absorbed by samples.
In high-energy experiments, particle therapy methods are applied while nuclear imaging and computed tomography are used to track transient radiation dose effects.
The synergy between magnetic resonance and computed tomography enhances our ability to characterize materials and accurately measure radiation dose in complex experiments.
By integrating nuclear imaging with magnetic resonance methods, we can simultaneously monitor particle therapy beams and record the associated radiation dose.
Our experimental setup employs computed tomography and nuclear imaging to provide detailed spatial resolution while we control the radiation dose during particle therapy experiments.
Advances in m

In [59]:
demo_dict['physics']['output']['keywords'][0][0]

'radiation'

In [60]:
demo_dict['physics']['output']['keywords_embeddings'][('radiation', '3df6a394f8b4931fdb2a8080c80873a1', 0)]

array([ 1.49912253e-01,  6.51724517e-01, -8.41086566e-01,  7.90924013e-01,
        1.03710294e+00, -3.86156052e-01,  4.73796397e-01,  2.21895024e-01,
       -4.36648875e-01, -1.17334950e+00,  4.55114007e-01, -1.08753942e-01,
        6.91481829e-02,  6.16102099e-01,  2.94042677e-01,  7.55264282e-01,
        7.96601474e-01, -2.67991394e-01,  4.75785881e-01,  8.21194530e-01,
       -5.61068095e-02, -8.51302445e-01,  2.73473233e-01,  8.83262873e-01,
        2.66881157e-02, -8.71792734e-01,  5.61894238e-01, -1.80056602e-01,
       -6.80439353e-01, -7.71421939e-02, -1.53119221e-01, -4.12053883e-01,
       -3.91803235e-01, -1.65946275e-01,  1.15248859e+00,  3.63837481e-01,
       -7.13789523e-01, -2.14867845e-01, -4.31918263e-01, -4.05234367e-01,
       -9.34219599e-01, -3.36175174e-01, -3.67999554e-01, -3.11957300e-01,
       -2.57999271e-01,  2.44418964e-01, -3.01977009e-01,  9.37250674e-01,
        3.34436566e-01, -5.27438745e-02, -5.15248775e-01,  5.00464797e-01,
        2.70253956e-01, -

In [61]:
demo_dict['physics']['output']['chunk_embeddings'][0]

tensor([-4.8073e-02,  5.3978e-01, -5.4517e-01,  7.4264e-01,  8.5021e-01,
        -2.8809e-01,  3.2309e-01, -4.9678e-03, -3.7681e-01, -9.9622e-01,
         3.0940e-01,  5.8799e-02,  3.0894e-01,  5.6276e-01,  2.6995e-01,
         9.4119e-01,  7.4284e-01, -4.2787e-01,  2.9425e-01,  7.5248e-01,
        -4.9144e-02, -7.9058e-01,  6.0117e-01,  7.4318e-01,  1.8561e-01,
        -3.8810e-01,  7.1707e-01, -5.1664e-01, -5.5897e-01,  3.7133e-02,
        -1.9045e-01, -3.7268e-01, -3.3250e-01, -5.0794e-01,  8.5774e-01,
         1.6944e-01, -6.5991e-01, -1.3306e-01, -5.4972e-01, -2.0446e-01,
        -8.3461e-01, -9.6582e-02, -4.7371e-01, -2.6865e-01, -5.1216e-01,
         3.9162e-01, -2.4867e-01,  7.4361e-01,  1.9034e-01, -2.5714e-01,
        -4.9354e-01,  6.5841e-01,  5.8387e-01, -4.0882e-01, -6.8096e-01,
         5.5246e-01,  1.3330e-01, -4.0379e-01, -3.1711e-01, -1.1502e+00,
         2.1215e-01, -2.8580e-02, -1.3919e-01, -4.6030e-02,  1.5633e-01,
        -3.4431e-01,  5.0543e-02,  6.9246e-01, -8.8

In [62]:
demo_dict['physics']['output']['averaged_keywords_embeddings'][0]

array([ 1.03707472e-02,  5.15271783e-01, -5.98248899e-01,  7.74544716e-01,
        8.92845869e-01, -3.87228370e-01,  3.36110920e-01,  5.90611212e-02,
       -3.84960771e-01, -1.06577790e+00,  3.85266364e-01,  5.47826174e-04,
        2.46131182e-01,  4.75205064e-01,  3.31979781e-01,  1.01695955e+00,
        7.37062335e-01, -2.29022816e-01,  1.84303418e-01,  7.31678843e-01,
       -1.07785910e-01, -8.78324747e-01,  5.96780717e-01,  8.33316147e-01,
        1.38645649e-01, -6.11969352e-01,  6.09405994e-01, -3.71135473e-01,
       -7.29443908e-01,  9.92265195e-02, -7.55077302e-02, -3.51346552e-01,
       -4.48665768e-01, -3.79470915e-01,  1.03870070e+00,  1.96237057e-01,
       -8.97389591e-01, -1.19770490e-01, -4.84234333e-01, -2.82709599e-01,
       -9.16517377e-01, -2.07716361e-01, -4.61855114e-01, -1.76912695e-01,
       -3.79636645e-01,  3.76896590e-01, -2.26465508e-01,  6.98122978e-01,
        2.68193483e-01, -1.94359452e-01, -5.66117704e-01,  6.82561219e-01,
        4.82355952e-01, -

In [63]:
print(demo_dict['medicine']['input'])


In modern oncology, accurate radiation dose delivery during particle therapy is often guided by magnetic resonance imaging to ensure precise tumor targeting.
Clinicians routinely combine computed tomography with nuclear imaging to evaluate both anatomical details and functional aspects when planning a treatment strategy that minimizes radiation dose.
Magnetic resonance imaging is used alongside computed tomography to refine treatment planning, ensuring that the prescribed radiation dose for radiotherapy is both accurate and safe.
The integration of nuclear imaging and magnetic resonance techniques helps physicians monitor radiation dose in real time during advanced particle therapy sessions.
In pediatric radiology, computed tomography protocols are optimized by combining nuclear imaging feedback with meticulous radiation dose control to reduce overall exposure.
Physicians employ computed tomography and nuclear imaging together to assess disease progression while carefully adjusting th

In [64]:
demo_dict['medicine']['output']['keywords'][0][1]

'radiation'

In [65]:
demo_dict['medicine']['output']['keywords_embeddings'][('radiation', 'dc2dd9513fcaa5d7ae882c1281b8e9a9', 0)]

array([ 5.14723212e-02,  3.54533702e-01, -5.49135327e-01,  1.70208722e-01,
        9.62560236e-01, -2.55237639e-01,  6.11409605e-01,  3.97096276e-01,
       -5.98235548e-01, -1.34184909e+00,  6.45920694e-01, -1.10128999e-01,
       -1.42805681e-01,  4.41548049e-01,  4.28974092e-01,  7.10932851e-01,
        8.07587862e-01,  4.03699242e-02,  3.02973390e-01,  8.76066625e-01,
       -2.26393476e-01, -1.23120582e+00,  4.51097041e-01,  9.22523499e-01,
       -3.72873276e-01, -5.07088184e-01,  2.34238073e-01,  9.50892568e-02,
       -8.25174928e-01, -2.72299439e-01, -3.32798064e-01, -5.32335460e-01,
       -6.05541408e-01,  1.55729130e-01,  1.33715856e+00,  4.04854536e-01,
       -8.20016861e-01, -3.03969532e-01, -5.06246269e-01, -5.91829419e-01,
       -6.61754727e-01, -5.15120208e-01, -7.18152165e-01, -6.21705055e-01,
        5.01031019e-02, -2.59910524e-01, -2.07026750e-01,  5.60898721e-01,
        9.54153955e-01, -5.70168048e-02, -8.07398498e-01,  6.47480488e-01,
        4.24717396e-01, -

In [66]:
demo_dict['medicine']['output']['chunk_embeddings'][0]

tensor([-9.0892e-02,  2.0606e-01, -1.3013e-01,  1.5803e-01,  7.7849e-01,
        -2.2041e-01,  6.0914e-01,  1.9348e-01, -3.9666e-01, -1.2385e+00,
         4.5728e-01,  1.6343e-01,  1.8968e-01,  2.7750e-01,  3.7921e-01,
         7.9647e-01,  8.1275e-01, -4.5980e-02,  1.3333e-02,  8.7047e-01,
        -1.8062e-01, -1.3728e+00,  6.1140e-01,  6.0221e-01, -3.2039e-01,
         1.0307e-01,  4.1112e-01, -2.9993e-01, -6.8649e-01, -1.8796e-01,
        -3.7927e-01, -4.2875e-01, -4.4342e-01, -1.7514e-01,  1.0359e+00,
         3.7746e-01, -7.2847e-01, -2.0577e-01, -5.6901e-01, -2.4843e-01,
        -6.5791e-01, -1.7418e-01, -7.2296e-01, -6.8471e-01, -1.6633e-01,
        -4.2752e-02, -1.8670e-01,  3.2607e-01,  8.0489e-01, -1.5333e-01,
        -8.5726e-01,  8.6638e-01,  5.1230e-01, -3.6231e-01, -7.0967e-01,
         5.0563e-01,  1.3303e-01, -5.3541e-01, -4.9296e-01, -8.7815e-01,
         1.5683e-01, -2.5069e-01,  1.4712e-02, -4.3133e-02,  9.7421e-02,
        -2.4077e-01, -2.2008e-01,  9.9318e-01, -5.9

In [67]:
demo_dict['medicine']['output']['averaged_keywords_embeddings'][0]

array([ 2.96364836e-02,  2.32107759e-01, -2.81814814e-01,  1.98495463e-01,
        8.77798140e-01, -3.18132281e-01,  5.95514059e-01,  2.44094744e-01,
       -4.38789845e-01, -1.26893020e+00,  5.61549306e-01,  2.29662471e-02,
        6.97472394e-02,  2.86283314e-01,  4.00312603e-01,  9.06026900e-01,
        8.21938097e-01,  1.97411031e-01, -4.49027456e-02,  8.28061283e-01,
       -2.74077713e-01, -1.40738928e+00,  5.86778581e-01,  7.97904372e-01,
       -3.29031706e-01, -1.38976991e-01,  1.87983245e-01, -1.46959752e-01,
       -8.22347701e-01, -6.90213442e-02, -2.69357920e-01, -4.58342075e-01,
       -5.62004149e-01,  1.43384347e-02,  1.25452971e+00,  3.92797500e-01,
       -8.83708596e-01, -2.31702819e-01, -5.84371865e-01, -3.88802409e-01,
       -6.02136314e-01, -3.05783957e-01, -7.47179449e-01, -5.20615876e-01,
       -8.12231377e-02, -1.21139973e-01, -2.10623637e-01,  3.01921785e-01,
        8.85978639e-01, -1.18786521e-01, -9.06107783e-01,  8.00657213e-01,
        4.62081909e-01, -